In [0]:
# ============================================================

#

# GOLD – dim_status

# Grain: (source_user_id, project, status_code)

#

# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.window import Window

# ---------------- CONFIG ----------------

SILVER_TICKETS_TABLE = "workspace.slainte_silver.easyvista_tickets_silver_demo"

GOLD_DB = "workspace.slainte_gold"

DIM_STATUS = f"{GOLD_DB}.dim_status"

# ---------------- SETUP ----------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

df_tickets = spark.table(SILVER_TICKETS_TABLE)

# ============================================================

# 1️⃣ Base status extraction

# ============================================================

df_status_base = (

    df_tickets

    .select(

        F.col("source_user_id"),

        F.col("project"),

        F.upper(F.trim(F.col("status"))).alias("status_code")

    )

    .filter(

        F.col("source_user_id").isNotNull() &

        F.col("project").isNotNull() &

        F.col("status_code").isNotNull()

    )

    .distinct()

)

# ============================================================

# 2️⃣ Human-readable status label

# ============================================================

df_status_base = df_status_base.withColumn(

    "status_label",

    F.initcap(F.regexp_replace(F.col("status_code"), "_", " "))

)

# ============================================================

# 3️⃣ Generate surrogate key

# ============================================================

status_window = Window.partitionBy(

    "source_user_id", "project"

).orderBy("status_code")

df_dim_status = (

    df_status_base

    .withColumn("status_id", F.row_number().over(status_window))

    .withColumn("created_at", F.current_timestamp())

    .select(

        "status_id",

        "source_user_id",

        "project",

        "status_code",

        "status_label"

    )

)

# ============================================================

# 4️⃣ Write GOLD dimension

# ============================================================

df_dim_status.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_STATUS)

# ============================================================

# 5️⃣ Preview

# ============================================================

print("✅ dim_status created successfully")

display(

    spark.table(DIM_STATUS)

         .orderBy("source_user_id", "project", "status_id")

)
 